# Diabetes Classification

In [121]:
import pandas
import numpy
# from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
data = pandas.read_csv('/content/DiabetesTraining.csv')
print(data.head())

   gender   age  hypertension  heart_disease smoking_history    bmi  \
0  Female  80.0             0              1           never  25.19   
1  Female  80.0             0              1           never  25.19   
2  Female  54.0             0              0         No Info  27.32   
3    Male  28.0             0              0           never  27.32   
4  Female  36.0             0              0         current  23.45   

   HbA1c_level  blood_glucose_level  diabetes  
0          6.6                  140         0  
1          6.6                  140         0  
2          6.6                   80         0  
3          5.7                  158         0  
4          5.0                  155         0  


## Solving for part (a)
Using the numerical features 'age', 'BMI', 'HbA1c level', and 'blood glucose level', find the vector (w) to be used for linear discriminant analysis for binary classification. Standardize the data first by dividing the feature values by respective standard deviation.

The binary classification being a diagnoses of diabetes, or no diabetes denoted by 1 and 0 respectively in the last column.

In [122]:
# First select the target features
features = ['age','bmi','HbA1c_level','blood_glucose_level', 'diabetes']
part_a_data = data[features]
# print(part_a_data.loc[:, 'age'])

# Standardize the data of said features
# Subtract the mean and divide by the standard deviation to remove bias and lessen outliers
standardized_data =( part_a_data.loc[:, features[:-1]]  - part_a_data.loc[:, features[:-1]].mean() )/ part_a_data.loc[:, features[:-1]].std()
# standardized_data = part_a_data.loc[:, features[:-1]] / part_a_data.loc[:, features[:-1]].std()
standardized_data = pandas.concat([standardized_data, part_a_data['diabetes']], axis=1)
# print(standardized_data)

# Divide rows into 2 sets, one for diabetes (1), another for no diabetes (0)
diabetes_data = standardized_data[standardized_data['diabetes'] == 1]
no_diabetes_data = standardized_data[standardized_data['diabetes'] == 0]
# print(diabetes_data)
# print(no_diabetes_data)

# Calculate the within-class scatter (this is the s_1 and s_2 for the equation J(w))
# scatter matrix also refers to the covariance matrix when dataset is normalized
sw1 = diabetes_data[features[:-1]].cov()
sw0 = no_diabetes_data[features[:-1]].cov()
# print(sw1)
# print(sw0)

# Get S_w by combining the 2 scatter matrices
S_W = sw1 + sw0
# print(S_W)

# Calculate the between-class scatter matrix (S_B = (m_1 - m_2)(m_1 - m_2)^T)
# Find the mean vector for the 2 sets (this is the mean for each feature arranged in a vector)
diabetes_mean = diabetes_data[features[:-1]].mean()
no_diabetes_mean = no_diabetes_data[features[:-1]].mean()
# print(total_mean)
# print(diabetes_mean)
# print(no_diabetes_mean)

# Find (m_1 - m_2)
sb1 = diabetes_mean - no_diabetes_mean
# print(sb1)

# S_B = sb1 * sb1^T
S_B = numpy.outer(sb1.values, sb1.values)
# print(S_B)

# Combine S_W^-1 * S_B, and then find eigenvalues and eigenvectors
SW_inv = numpy.linalg.inv(S_W.values)

# Can also directly solve for w
result_w_no_bcsm = SW_inv @ (diabetes_mean.values - no_diabetes_mean.values)
print("Final result for vector (w):", result_w_no_bcsm)

eigenvalues, eigenvectors = numpy.linalg.eig(SW_inv @ S_B)
# print(eigenvalues)
# print(eigenvectors)

# Sort eigenvectors by eigenvalues and choose the eigenvector corresponding to the largest eigenvalue as the as the Linear Discriminant
indices = numpy.argsort(eigenvalues)[::-1]
sorted_values = eigenvalues[indices]
sorted_vectors = eigenvectors[:, indices]
# print(sorted_values)
# print(sorted_vectors)

# Vector w should be the first result
result_w = sorted_vectors[:, 0]
print("Final result for vector (w) with Fisher's:", result_w)
print("\n\n")

Final result for vector (w): [0.46895534 0.30155618 0.9292657  0.58976012]
Final result for vector (w) with Fisher's: [-0.38009717 -0.24441698 -0.75318741 -0.47801172]





## Solving for part (b)
Compute the Gini impurity and Information gain for attributes 'hypertension' and 'heart disease'.


In [123]:
# Gini Impurity
# First get the data we are interested in
part_b_data = data[['hypertension', 'heart_disease', 'diabetes']]
# print(part_b_data)

# Gini impurity of diabetes
diabetes_count = part_b_data[part_b_data['diabetes'] == 1].shape[0]
no_diabetes_count = part_b_data[part_b_data['diabetes'] == 0].shape[0]
total_count = diabetes_count + no_diabetes_count
GI_root = 1 - (diabetes_count / total_count)**2 - (no_diabetes_count / total_count)**2
print("Gini Impurity for Root Node:", GI_root)

# Hypertension Gini Impurity
hypertension_data = part_b_data[part_b_data['hypertension'] == 1]
no_hypertension_data = part_b_data[part_b_data['hypertension'] == 0]

# Get number of diabetes patients in each branch
diabetes_hypertension = hypertension_data[hypertension_data['diabetes'] == 1].shape[0]
no_diabetes_hypertension = hypertension_data[hypertension_data['diabetes'] == 0].shape[0]

diabetes_no_hypertension = no_hypertension_data[no_hypertension_data['diabetes'] == 1].shape[0]
no_diabetes_no_hypertension = no_hypertension_data[no_hypertension_data['diabetes'] == 0].shape[0]

# Calculate the Gini Impurity
GI_hyper1 = 1 - (diabetes_hypertension / hypertension_data.shape[0])**2 - (no_diabetes_hypertension / hypertension_data.shape[0])**2
GI_hyper2 = 1 - (diabetes_no_hypertension / no_hypertension_data.shape[0])**2 - (no_diabetes_no_hypertension / no_hypertension_data.shape[0])**2
print("Gini Impurity for Hypertension:", GI_hyper1 + GI_hyper2)

Gini Impurity for Root Node: 0.14711045401222633
Gini Impurity for Hypertension: 0.49592446860977835


In [124]:
# Heart disease Gini impurity
heart_disease_data = part_b_data[part_b_data['heart_disease'] == 1]
no_heart_disease_data = part_b_data[part_b_data['heart_disease'] == 0]

# Get number of diabetes patients in each branch
diabetes_heart_disease = heart_disease_data[heart_disease_data['diabetes'] == 1].shape[0]
no_diabetes_heart_disease = heart_disease_data[heart_disease_data['diabetes'] == 0].shape[0]

diabetes_no_heart_disease = no_heart_disease_data[no_heart_disease_data['diabetes'] == 1].shape[0]
no_diabetes_no_heart_disease = no_heart_disease_data[no_heart_disease_data['diabetes'] == 0].shape[0]

# Calculate the Gini Impurity
GI_heart1 = 1 - (diabetes_heart_disease / heart_disease_data.shape[0])**2 - (no_diabetes_heart_disease / heart_disease_data.shape[0])**2
GI_heart2 = 1 - (diabetes_no_heart_disease / no_heart_disease_data.shape[0])**2 - (no_diabetes_no_heart_disease / no_heart_disease_data.shape[0])**2
print("Gini Impurity for Heart Disease:", GI_heart1 + GI_heart2)

Gini Impurity for Heart Disease: 0.4593963219947619


In [125]:
# Information Gain
# Calculate Total Entropy
diabetes_count = part_b_data[part_b_data['diabetes'] == 1].shape[0]
no_diabetes_count = part_b_data[part_b_data['diabetes'] == 0].shape[0]
total_count = diabetes_count + no_diabetes_count
# print("Total Count:", total_count)
# print("Diabetes Count:", diabetes_count)
# print("No Diabetes Count:", no_diabetes_count)
Entropy_diabetes = - ((diabetes_count / total_count) * numpy.log2(diabetes_count / total_count) + (no_diabetes_count / total_count) * numpy.log2(no_diabetes_count / total_count))
print("Diabetes Entropy:", Entropy_diabetes)

# Calculate Information Gain for Hypertension
Entropy_hypertension = -1 * ((diabetes_hypertension/hypertension_data.shape[0]) * numpy.log2(diabetes_hypertension/hypertension_data.shape[0]) + (no_diabetes_hypertension/hypertension_data.shape[0]) * numpy.log2(no_diabetes_hypertension/hypertension_data.shape[0]))
Entropy_no_hypertension = -1 * ((diabetes_no_hypertension/no_hypertension_data.shape[0]) * numpy.log2(diabetes_no_hypertension/no_hypertension_data.shape[0]) + (no_diabetes_no_hypertension/no_hypertension_data.shape[0]) * numpy.log2(no_diabetes_no_hypertension/no_hypertension_data.shape[0]))
INFO_gain_hypertension = Entropy_diabetes - ((hypertension_data.shape[0] / part_b_data.shape[0]) * Entropy_hypertension + (no_hypertension_data.shape[0] / part_b_data.shape[0]) * Entropy_no_hypertension)
print("Information Gain for Hypertension:", INFO_gain_hypertension)

Diabetes Entropy: 0.40199136425000853
Information Gain for Hypertension: 0.014297005653686634


In [126]:
# Calculate Information Gain for Heart Disease
Entropy_heart_disease = -1 * ((diabetes_heart_disease/heart_disease_data.shape[0]) * numpy.log2(diabetes_heart_disease/heart_disease_data.shape[0]) + (no_diabetes_heart_disease/heart_disease_data.shape[0]) * numpy.log2(no_diabetes_heart_disease/heart_disease_data.shape[0]))
Entropy_no_heart_disease = -1 * ((diabetes_no_heart_disease/no_heart_disease_data.shape[0]) * numpy.log2(diabetes_no_heart_disease/no_heart_disease_data.shape[0]) + (no_diabetes_no_heart_disease/no_heart_disease_data.shape[0]) * numpy.log2(no_diabetes_no_heart_disease/no_heart_disease_data.shape[0]))
INFO_gain_heart_disease = Entropy_diabetes - ((heart_disease_data.shape[0] / part_b_data.shape[0]) * Entropy_heart_disease + (no_heart_disease_data.shape[0] / part_b_data.shape[0]) * Entropy_no_heart_disease)
print("Information Gain for Heart Disease:", INFO_gain_heart_disease)

Information Gain for Heart Disease: 0.003982146278906973


### Conclusion
When calculating for the Gini Impurity for deciding whether to split on hypertension or heart disease at the beginning of the decision tree, heart disease has a lower Gini Impurity. However, when calculating for the information gain hypertension has a better value.